In [0]:
%sql
USE CATALOG pysparkdbt;

In [0]:
%sql
SHOW TABLES IN gold;


database,tableName,isTemporary
gold,customer_summary,false
gold,dimcustomers,false
gold,dimdrivers,false
gold,dimlocations,false
gold,dimpayments,false
gold,dimvehicles,false
gold,driver_summary,false
gold,facttrips,false
gold,monthly_revenue,false
gold,payment_summary,false


### Top 10 customers

In [0]:
%sql
SELECT
    customer_id,
    full_name,
    lifetime_spend,
    total_trips
FROM gold.customer_summary
ORDER BY lifetime_spend DESC
LIMIT 10;

customer_id,full_name,lifetime_spend,total_trips
126,Richard Rosales,783.81,11
6,Melissa Blair,668.3299999999999,9
44,Joshua Brown,619.61,10
59,Pamela Rush,602.9,9
194,Maria Ramsey,588.02,10
153,Brian Williams,571.07,8
3,Samuel Rodriguez,551.64,9
62,Tara Roberts,541.48,9
103,Stephen Sanders,528.34,8
80,Toni Morris,511.40999999999997,8


### Drivers rank by revenue

In [0]:
%sql
SELECT
    driver_id,
    full_name,
    ROUND(total_revenue,2),
    RANK() OVER (
        ORDER BY total_revenue DESC
    ) AS revenue_rank
FROM gold.driver_summary;

driver_id,full_name,"ROUND(total_revenue,2)",revenue_rank
39,Karen Williamson,1507.88,1
47,Sherry Hartman,1492.75,2
4,Theresa Benson,1484.58,3
22,Melissa Erickson,1379.35,4
6,Debra Smith,1361.5,5
27,Antonio Armstrong,1352.85,6
20,Jon Cole,1325.91,7
8,Todd Young,1324.7,8
36,Jessica Bradley,1315.67,9
18,Lisa Duarte,1309.28,10


### Customers belonging to Bronze, Silver and Gold segments

In [0]:
%sql
SELECT
    customer_id,
    full_name,
    lifetime_spend,

    CASE
        WHEN lifetime_spend >= 5000 THEN 'Gold'
        WHEN lifetime_spend >= 2500 THEN 'Silver'
        ELSE 'Bronze'
    END AS customer_segment

FROM gold.customer_summary;

customer_id,full_name,lifetime_spend,customer_segment
124,Nathan Huang,381.78999999999996,Bronze
14,Patricia Little,248.39,Bronze
162,Ryan Nelson,123.86999999999999,Bronze
186,Mary Brown,502.29999999999995,Bronze
190,Jonathan Tran,353.77000000000004,Bronze
2,Jonathan Hansen,18.88,Bronze
29,Jacqueline Perez,251.93,Bronze
43,Meghan Ibarra,274.07,Bronze
64,Caitlyn Rose,195.64,Bronze
65,Jennifer Williams,252.22,Bronze


### revenue changed month-over-month

In [0]:
%sql
SELECT
    month,
    ROUND(revenue,2),

    ROUND(LAG(revenue) OVER(
        ORDER BY month
    ),2) AS previous_month,

    ROUND(revenue -
    LAG(revenue) OVER(
        ORDER BY month
    ),2) AS revenue_growth

FROM gold.monthly_revenue;

month,"ROUND(revenue,2)",previous_month,revenue_growth
2025-06-01T00:00:00.000Z,4868.86,null,null
2025-07-01T00:00:00.000Z,16001.19,4868.86,11132.33
2025-08-01T00:00:00.000Z,17992.11,16001.19,1990.92
2025-09-01T00:00:00.000Z,12260.19,17992.11,-5731.92


### cumulative revenue over time

In [0]:
%sql
SELECT
    month,
    ROUND(revenue,2),

    ROUND(SUM(revenue) OVER(
        ORDER BY month
    ),2) AS cumulative_revenue

FROM gold.monthly_revenue;

month,"ROUND(revenue,2)",cumulative_revenue
2025-06-01T00:00:00.000Z,4868.86,4868.86
2025-07-01T00:00:00.000Z,16001.19,20870.05
2025-08-01T00:00:00.000Z,17992.11,38862.16
2025-09-01T00:00:00.000Z,12260.19,51122.35


### Payment methods contribute the most revenue

In [0]:
%sql
SELECT
    payment_method,
    ROUND(revenue,2),

    ROUND(
        revenue * 100 /
        SUM(revenue) OVER(),
        2
    ) AS revenue_percentage

FROM gold.payment_summary
ORDER BY revenue DESC;

payment_method,"ROUND(revenue,2)",revenue_percentage
Wallet,6229.55,12.26
Card,6144.75,12.09
Wallet,5947.2,11.7
Card,5691.69,11.2
Cash,5684.9,11.18
Cash,5491.09,10.8
Wallet,5313.51,10.45
Card,5281.8,10.39
Cash,5044.68,9.92


### Top 3 drivers in each city

In [0]:
%sql
SELECT *
FROM (
    SELECT
        city,
        full_name,
        ROUND(total_revenue,2),

        ROW_NUMBER() OVER(
            PARTITION BY city
            ORDER BY total_revenue DESC
        ) AS city_rank

    FROM gold.driver_summary
)
WHERE city_rank <= 3;

city,full_name,"ROUND(total_revenue,2)",city_rank
Brownburgh,Karen Jensen,625.46,1
Charlesborough,Sarah Ross,1215.03,1
Christianshire,Anna Wells,1075.58,1
Clementsmouth,Stephanie Perry,577.37,1
Cynthiachester,Gregory Stevenson,1017.28,1
Deborahhaven,Howard Hickman,897.5,1
East Brendatown,Christopher Foley,1276.7,1
East Dorothy,Latasha Lopez,1024.01,1
East Katherine,Daniel Hill,1262.08,1
Ericmouth,Nicholas Williamson,860.27,1


### Most revenue based on vehicle types

In [0]:
%sql
SELECT
    vehicle_type,
    ROUND(total_revenue,2),

    DENSE_RANK() OVER(
        ORDER BY total_revenue DESC
    ) AS revenue_rank

FROM gold.vehicle_summary;

vehicle_type,"ROUND(total_revenue,2)",revenue_rank
Sedan,1507.88,1
Sedan,1492.75,2
Van,1484.58,3
Hatchback,1379.35,4
SUV,1361.5,5
Van,1352.85,6
Luxury,1325.91,7
Sedan,1324.7,8
Van,1315.67,9
Luxury,1309.28,10


### Revenue summary with grand total

In [0]:
%sql
SELECT
    payment_method,
    ROUND(SUM(revenue),2) AS total_revenue

FROM gold.payment_summary

GROUP BY ROLLUP(payment_method);

payment_method,total_revenue
Card,17118.24
Wallet,17490.26
Cash,16220.67
null,50829.17
